# Metadata extraction from rental agreements
Extracts **Agreement Value, Start Date, End Date, Renewal Notice (days), Party One, Party Two** from `.docx` and scanned `.png` files using an LLM (no regex / rule-based logic).

Run the cells top to bottom. Documents that were already processed are read from `outputs/cache/`, so re-running costs no API calls.

In [ ]:
import os, sys
from pathlib import Path

# make the notebook work whether it is started from the project root or from notebooks/
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
from src.run import predict_folder
from src.evaluate import load_labels, per_field_recall, mismatches

ROOT = Path.cwd()
labels = load_labels(ROOT / "data" / "train.csv", ROOT / "data" / "test.csv")
pd.set_option("display.width", 200)
print("Project root:", ROOT)

## 1. Predictions on the test set

In [ ]:
pred_test = predict_folder(ROOT / "data" / "test", only=set(labels.index))
(ROOT / "outputs").mkdir(exist_ok=True)
pred_test.to_csv(ROOT / "outputs" / "predictions_test.csv")
pred_test

## 2. Per-field recall on the test set
*Strict* = exact string match (the assignment metric). *Lenient* ignores letter case and extra spaces.

In [ ]:
recall_test = per_field_recall(pred_test, labels)
recall_test.to_csv(ROOT / "outputs" / "recall_test.csv", index=False)
recall_test

## 3. Validation on the labelled training files

In [ ]:
pred_train = predict_folder(ROOT / "data" / "train", only=set(labels.index))
pred_train.to_csv(ROOT / "outputs" / "predictions_train.csv")
recall_train = per_field_recall(pred_train, labels)
recall_train.to_csv(ROOT / "outputs" / "recall_train.csv", index=False)
recall_train

## 4. Where the predictions differ from the labels

In [ ]:
pd.concat([mismatches(pred_test, labels).assign(Split="test"),
           mismatches(pred_train, labels).assign(Split="train")], ignore_index=True)

## 5. Using the REST API (optional)
Start the server in a terminal first: `uvicorn api.main:app --reload`

In [ ]:
import requests
sample = ROOT / "data" / "test" / "24158401-Rental-Agreement.png"
try:
    with open(sample, "rb") as f:
        r = requests.post("http://127.0.0.1:8000/extract", files={"file": (sample.name, f)}, timeout=180)
    print(r.status_code)
    print(r.json())
except requests.exceptions.ConnectionError:
    print("API server is not running - start it with: uvicorn api.main:app --reload")